# 05 — Intervalles de confiance & CQR

**Question centrale :** Comment donner une incertitude honnête à nos prédictions ?

## Le problème de la confiance aveugle

Un modèle ML prédit un nombre. Mais ce nombre est faux d'une manière ou d'une autre.
La question n'est pas "est-ce que la prédiction est juste ?" mais "à quelle distance du vrai prix peut-elle être ?"

Sans intervalle de confiance, un agriculteur qui voit "prix prédit = 4.50 USD/bu dans 20j" peut :
- Vendre maintenant à 4.20 et rater 0.30$/bu si le modèle avait raison
- Stocker en espérant 4.50 et vendre à 3.80 si le modèle avait tort

**Avec un intervalle :**
- Q10 = 3.67, Q50 = 4.06, Q90 = 4.50 → l'agriculteur sait que l'incertitude est énorme
- → décision prudente : vendre par tiers

## Pourquoi CQR et pas des intervalles bootstrap ?

| Méthode | Avantage | Problème |
|---|---|---|
| Bootstrap | Simple | Sous-estime l'incertitude en distribution shift |
| Bayésien | Incertitude modèle | Très lent, mal calibré sur données TS |
| Quantile regression | Direct | Pas de garantie de couverture |
| **CQR** | **Garantie couverture ≥ 90%** | **Besoin d'un calibration set** |

La CQR (Conformalized Quantile Regression) donne une **garantie mathématique** :
P(y_true ∈ [Q10, Q90]) ≥ 90% sur les données de calibration.

In [ ]:
import sys
sys.path.insert(0, '../../src')
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

ROOT = Path('../../')
plt.style.use('seaborn-v0_8-whitegrid')

cqr  = pd.read_parquet(ROOT / 'artefacts/professional_study/cqr_results.parquet')
cal  = pd.read_parquet(ROOT / 'artefacts/professional_study/calibrated_predictions.parquet')
ss   = json.loads((ROOT / 'artefacts/professional_study/study_summary.json').read_text())

print('CQR columns :', list(cqr.columns))
print('Cal columns :', list(cal.columns)[:15])
print(f'CQR rows : {len(cqr):,}  |  Cal rows : {len(cal):,}')

## 1. Couverture empirique par horizon

On vérifie que la promesse est tenue : au moins 90% des vraies valeurs tombent dans l'intervalle.

In [ ]:
cqr_summary = ss.get('cqr_summary', [])

if cqr_summary:
    cqr_df = pd.DataFrame(cqr_summary)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Couverture
    colors = ['green' if v >= 0.90 else 'red' for v in cqr_df['coverage']]
    axes[0].bar(cqr_df['horizon'].astype(str), cqr_df['coverage'] * 100, color=colors, alpha=0.8)
    axes[0].axhline(90, color='red', lw=2, ls='--', label='Objectif 90%')
    axes[0].axhline(95, color='orange', lw=1, ls=':', label='95% (conservateur)')
    axes[0].set_title('Couverture CQR empirique par horizon (objectif ≥90%)')
    axes[0].set_xlabel('Horizon (jours)')
    axes[0].set_ylabel('Couverture (%)')
    axes[0].set_ylim(80, 100)
    axes[0].legend()
    
    for i, (_, row) in enumerate(cqr_df.iterrows()):
        axes[0].annotate(f"{row['coverage']:.1%}",
                        xy=(i, row['coverage']*100 + 0.3), ha='center', fontweight='bold')
    
    # Largeur des intervalles
    axes[1].bar(cqr_df['horizon'].astype(str), cqr_df['width'], color='steelblue', alpha=0.8)
    axes[1].set_title('Largeur moyenne des intervalles CQR [Q10, Q90]\n(en log-return)')
    axes[1].set_xlabel('Horizon (jours)')
    axes[1].set_ylabel('Largeur (log-return)')
    
    for i, (_, row) in enumerate(cqr_df.iterrows()):
        axes[1].annotate(f"{row['width']:.3f}",
                        xy=(i, row['width'] + 0.003), ha='center')
    
    plt.tight_layout()
    plt.show()
    print(cqr_df[['horizon','coverage','width','n']].to_string(index=False))

## 2. Distribution des prédictions et intervalles

In [ ]:
if 'covered' in cqr.columns and 'horizon' in cqr.columns:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    for ax, h in zip(axes.flat, sorted(cqr['horizon'].unique())):
        sub = cqr[cqr['horizon'] == h]
        covered = sub[sub['covered'] == True] if 'covered' in sub else sub
        not_cov = sub[sub['covered'] == False] if 'covered' in sub else pd.DataFrame()
        
        cov_rate = sub['covered'].mean() if 'covered' in sub.columns else float('nan')
        
        # Scatter : vrai vs. prédit
        if 'y_true' in sub.columns and 'q50' in sub.columns:
            ax.scatter(sub['q50'], sub['y_true'], alpha=0.15, s=3, color='steelblue', label='Couvert')
            lim = max(abs(sub['y_true'].max()), abs(sub['y_true'].min()))
            ax.plot([-lim, lim], [-lim, lim], 'r--', lw=1)
            ax.set_xlabel('Prédiction Q50')
            ax.set_ylabel('Vraie valeur')
        
        ax.set_title(f'H={h}j — Couverture = {cov_rate:.1%}', fontsize=11)
    
    plt.suptitle('Prédictions CQR : vraies valeurs vs. quantile médian', fontsize=13)
    plt.tight_layout()
    plt.show()

## 3. Comparaison CQR vs. Split-Conformal

On a testé deux approches de prédiction conforme :

- **CQR** (Conformalized Quantile Regression) : modèle de quantile + calibration adaptative
- **Split-Conformal** : résidus du modèle principal + expansion par quantile

| Méthode | Couverture | Avantage |
|---|---|---|
| CQR | **91.7%** | Adaptatif, respecte la structure de la distribution |
| Split-Conformal | **89.0%** | Simple, rapide |

**Pourquoi CQR gagne :** il modélise asymétriquement l'incertitude (les hausses et baisses extrêmes ont des largeurs différentes).

In [ ]:
# Comparaison couvertures
cqr_mean = ss.get('cqr_mean_coverage', None)
sc_mean  = ss.get('split_conformal_mean_coverage', None)

if cqr_mean and sc_mean:
    fig, ax = plt.subplots(figsize=(8, 5))
    methods = ['Split-Conformal', 'CQR (notre choix)']
    coverages = [sc_mean * 100, cqr_mean * 100]
    bars = ax.bar(methods, coverages, color=['#d9534f', '#5cb85c'], alpha=0.85, width=0.4)
    ax.axhline(90, color='black', lw=2, ls='--', label='Objectif 90%')
    ax.set_ylim(85, 95)
    ax.set_ylabel('Couverture (%)')
    ax.set_title('CQR vs. Split-Conformal — couverture empirique moyenne')
    ax.legend()
    for bar, val in zip(bars, coverages):
        ax.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width()/2, val + 0.1),
                    ha='center', fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.show()

## 4. Interprétation de la largeur des intervalles

Un intervalle de largeur 0.25 en log-return à H=20j, qu'est-ce que ça veut dire en pratique ?

In [ ]:
# Traduction en USD/bu pour un prix de référence de 4.20 USD/bu
prix_ref = 4.20

print('Traduction des intervalles CQR en USD/bu (prix référence = 4.20 USD/bu)')
print('='*70)
for row in cqr_summary:
    h = row['horizon']
    w = row['width']
    # Interval [Q10, Q90] en logret = [-w/2, +w/2] approximativement
    q10_price = prix_ref * np.exp(-w/2)
    q90_price = prix_ref * np.exp(+w/2)
    spread = q90_price - q10_price
    print(f'H={h:2d}j  Intervalle = [{q10_price:.2f}, {q90_price:.2f}] USD/bu  '
          f'(spread = {spread:.2f} USD/bu = {spread/prix_ref:.0%} du prix)')

print()
print('Interprétation :')
print('- H=5j  : incertitude ~12%  → décision à court terme possible')
print('- H=20j : incertitude ~27%  → SELL_THIRDS recommandé (trop large pour parier)')
print('- H=30j : incertitude ~34%  → stockage risqué sauf si premium suffisant')

## 5. Ce qu'il faut améliorer

### Problème actuel : intervalles trop larges

La couverture est bonne (91.7%) mais les intervalles sont larges → ils disent souvent "je ne sais pas".

### Pistes pour affiner

1. **CQR par régime** — en régime range (90% du temps), l'incertitude est plus faible
2. **CQR par saison** — en été (volatilité max), élargir ; en hiver (volatilité min), réduire
3. **CQR sur le prix cash** — pas sur le log-return CBOT (plus directement actionnable)
4. **Modèle de base plus fort** — si Q10/Q90 initial est meilleur, l'intervalle CQR sera plus fin
5. **Données supplémentaires** — Crop Progress réduit l'incertitude en été

**→ Prochain carnet :** Régimes de marché — y a-t-il des états du marché détectables ?